# LangGraph Short-Term Memory (STM)

Demonstration of **Short-Term Memory** in LangGraph using `MemorySaver` and `thread_id`.

In [4]:
from typing import Annotated, TypedDict
from dotenv import load_dotenv, find_dotenv
from langchain_groq import ChatGroq
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, SystemMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver

load_dotenv(find_dotenv())

True

In [5]:
# 1. Define State
class chatstate(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [6]:
# 2. Initialize LLM
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.7,
)

In [7]:
# 3. Define chat_node
def chat_node(state: chatstate):
    # state["messages"] contains all messages in memory for this thread_id
    response = llm.invoke(state["messages"])
    # Return a list [response] for the add_messages reducer
    return {"messages": [response]}

In [8]:
# 4. Build and Compile Graph with Checkpointer (Short-Term Memory)
builder = StateGraph(chatstate)
builder.add_node("chat_node", chat_node)
builder.add_edge(START, "chat_node")
builder.add_edge("chat_node", END)

checkpointer = MemorySaver()
app = builder.compile(checkpointer=checkpointer)

In [9]:
# 5. Turn 1 (Thread 1)
config = {"configurable": {"thread_id": "thread_1"}}

res1 = app.invoke(
    {"messages": [("user", "Hi, my name is Shreyas and I love building agentic workflows with LangGraph.")]},
    config=config,
)
print(res1["messages"][-1].content)

Hey Shreyas! 👋 Great to meet a fellow LangGraph enthusiast.  

Building agentic workflows can be a lot of fun—and a powerful way to let LLMs take the wheel on complex tasks. What kind of projects are you working on, or are you just getting started and looking for ideas and best‑practice tips?

Here are a few things that often come up when people dive into LangGraph:

| Topic | Quick Tips / Resources |
|-------|------------------------|
| **State Management** | • Keep the state schema as flat as possible; nested dicts can get messy when you need to serialize/deserialize.<br>• Use `pydantic` models for strong‑typed state – LangGraph plays nicely with them. |
| **Node Design** | • Make each node **single‑purpose** (e.g., “fetch data”, “summarize”, “decide next step”). This keeps the graph easy to reason about and test.<br>• Return a `NodeResult` with both `output` and any `metadata` you might need downstream (e.g., timestamps, token usage). |
| **Looping & Conditional Branches** | • Lever

In [10]:
# 6. Turn 2 (Thread 1 - Testing Memory Retention)
res2 = app.invoke(
    {"messages": [("user", "What is my name and what do I love?")]},
    config=config,
)
print(res2["messages"][-1].content)

Your name is **Shreyas**, and you love **building agentic workflows with LangGraph**.


In [11]:
config_thread_2 = {"configurable": {"thread_id": "thread_2"}}

res3 = app.invoke(
    {"messages": [("user", "Do you know my name?")]},
    config=config_thread_2,
)
print(res3["messages"][-1].content)

I don’t have any information about you, so I don’t know your name. If you’d like to share it, feel free—otherwise we can continue our conversation without it!


In [15]:
# 8. Inspect State Snapshot and History
snapshot = app.get_state(config)
print("=== Messages Stored in Thread 1 Memory ===")
for i, m in enumerate(snapshot.values["messages"], 1):
    print(f"[{i}] {m.__class__.__name__}: {m.content}")

=== Messages Stored in Thread 1 Memory ===
[1] HumanMessage: Hi, my name is Shreyas and I love building agentic workflows with LangGraph.
[2] AIMessage: Hey Shreyas! 👋 Great to meet a fellow LangGraph enthusiast.  

Building agentic workflows can be a lot of fun—and a powerful way to let LLMs take the wheel on complex tasks. What kind of projects are you working on, or are you just getting started and looking for ideas and best‑practice tips?

Here are a few things that often come up when people dive into LangGraph:

| Topic | Quick Tips / Resources |
|-------|------------------------|
| **State Management** | • Keep the state schema as flat as possible; nested dicts can get messy when you need to serialize/deserialize.<br>• Use `pydantic` models for strong‑typed state – LangGraph plays nicely with them. |
| **Node Design** | • Make each node **single‑purpose** (e.g., “fetch data”, “summarize”, “decide next step”). This keeps the graph easy to reason about and test.<br>• Return a `Node